# PatchTCN — 端到端量价时序预测

**提交清单（4 个文件）**：`submit.ipynb` / `train.py` / `predict.py` / `model_weights.json`

**模型**：因果空洞卷积（TCN），输入为 1 分钟原始字段构成的日内 patch 序列。
无循环结构、无外部预训练权重。窗口与网络规模由 `train.py` 顶部常量决定，
推理时从权重文件读取，二者一致。随机种子固定（`train.SEED`）。

**输入合规**：仅使用主办方原始字段（25 个，上限 100）；预处理只含缺失值填充、
按字段固定单位换算、log1p，以及仅在训练区间拟合的标准化。
未使用收益率、价差、盘口不平衡、滚动统计、技术指标或降维。

**表名不硬编码**：`bar1m` 在公榜与私榜指向不同的物理表，因此训练与推理都从
`datasources` 取表名（官方《常见报错的自检方式》明确要求，硬编码会查到旧表）。

**公榜 / 私榜**：公榜直接加载随包提交的权重推理；私榜阶段平台按本包的 `train.py`
从零重训。下面的 `main` 在找不到权重时会自行触发重训，并把 `datasources` 里的
表名传进训练流程，不依赖平台以特定方式调用训练脚本。

In [ ]:
import os

import predict
import train

# Resolve the weight file next to these sources, not against the working
# directory: notebook code may be executed from elsewhere while train.py writes
# beside its own file, which would split the write and read paths.
MODEL_PATH = train.DEFAULT_WEIGHTS


def resolve_table(datasources):
    """Take the bar table from datasources; never hard-code it.

    bar1m points at different physical tables on the public and private boards,
    so a hard-coded name reads the wrong table (or nothing) during the private
    retrain.
    """
    if isinstance(datasources, dict):
        return datasources.get("bar1m") or next(iter(datasources.values()))
    return datasources


def ensure_weights(table):
    """Train from scratch if no weights are present.

    The private board discards the submitted weights and expects a retrain from
    this script, so triggering it here makes the package self-sufficient however
    the platform invokes it. With weights present (public board) this is a no-op.
    """
    if os.path.exists(MODEL_PATH):
        print(f"[submit] using existing weights: {MODEL_PATH}", flush=True)
        return MODEL_PATH
    print(f"[submit] no weights at {MODEL_PATH}; retraining on table={table}", flush=True)
    train.train_and_save(
        table=table,
        save_path=MODEL_PATH,
        start=train.TRAIN_START,
        end=train.TRAIN_END,
        is_local=False,
    )
    print(f"[submit] retrain done: {MODEL_PATH}", flush=True)
    return MODEL_PATH


def main(datasources, start_date, end_date):
    """平台入口：返回仅含 date / instrument / score 三列的 DataFrame。"""
    table = resolve_table(datasources)
    print(f"[submit] datasources={datasources} -> table={table}", flush=True)
    ensure_weights(table)
    return predict.main(datasources, start_date, end_date, model_path=MODEL_PATH)